# الدرس 05 - RAG الوكالية


## الإعداد

يعرض هذا الدفتر نمط Agentic RAG (التوليد المعزز بالاسترجاع) باستخدام إطار عمل Microsoft Agent.

**المتطلبات الأساسية:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — نقطة نهاية خدمة البحث Azure AI الخاصة بك
- `AZURE_SEARCH_API_KEY` — مفتاح API الخاص بخدمة البحث Azure AI لديك
- نشر Azure OpenAI مُكوّن عبر متغيرات البيئة
- تم تسجيل الدخول إلى Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ما هو Agentic RAG؟

يتبع RAG التقليدي مسارًا ثابتًا: استرجاع الوثائق، ثم توليد رد. **Agentic RAG** يذهب أبعد من ذلك عبر منح الوكيل القدرة على اتخاذ القرار بشأن **متى** و **كيف** يسترجع المعلومات.

مع Agentic RAG، يمكن للوكيل:
- **تحديد** ما إذا كان الاسترجاع مطلوبًا قبل الإجابة على السؤال
- **اختيار** مصدر البيانات أو الأداة التي سيتم الاستعلام منها
- **تقييم** النتائج المسترجعة وأداء استرجاعات متابعة إذا كانت المحاولة الأولى غير كافية
- **دمج** المعلومات من عدة خطوات استرجاع في إجابة متماسكة

هذا يجعل الوكيل أكثر مرونة ودقة مقارنة بمسار استرجاع-ثم-توليد ثابت.


## إنشاء أداة بحث

في Agentic RAG، يتم تغليف مصادر البيانات الخارجية كـ **أدوات** يمكن للوكيل استدعاؤها عند الطلب. هذا يسمح للوكيل بمعاملة الاسترجاع كإجراء آخر يمكنه اتخاذه، بدلاً من كونه خطوة إلزامية.

فيما يلي نحدد قاعدة معرفية للسفر ونعرضها كأداة يمكن للوكيل استخدامها للبحث عن معلومات الوجهة.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## بناء وكيل RAG

الآن نقوم بإنشاء وكيل يتم توجيهه لـ **استرجاع المعلومات دائمًا قبل الإجابة**. يستخدم الوكيل أداة `search_travel_knowledge` لتأسيس إجاباته على قاعدة المعرفة بدلاً من الاعتماد على بيانات تدريبه الخاصة.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## الاسترجاع التكراري — نمط الصانع-المدقق

الميزة الرئيسية في Agentic RAG هي **الاسترجاع التكراري**. يمكن للوكيل إجراء جولات متعددة من البحث للتحقق أو تحسين أو توسيع نتائجه الأولية — مشابهًا لتدفق عمل "الصانع-المدقق":

1. **خطوة الصانع**: يسترجع الوكيل المعلومات الأولية ويعد مسودة رد.
2. **خطوة المدقق**: يقوم الوكيل بإجراء استرجاعات إضافية للتحقق من التفاصيل أو ملء الثغرات.

في الأسفل، يُطلب من الوكيل سؤال يتطلب مقارنة وجهات متعددة، مما يدفعه إلى البحث عدة مرات.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## الملخص

في هذا الدرس تعلمت كيفية بناء نظام **Agentic RAG** باستخدام إطار عمل الوكيل من مايكروسوفت:

- يتيح **Agentic RAG** للوكلاء اتخاذ قرار تلقائي حول متى يسترجعون المعلومات، مما يجعل الاسترجاع ديناميكياً بدلاً من أن يكون ثابتًا.
- **الأدوات كمصادر بيانات**: تُغلف قواعد المعرفة الخارجية (مثل Azure AI Search) كأدوات يمكن للوكيل استدعاؤها.
- **الاسترجاع التكراري**: نمط الصانع-المدقق يمكّن الوكيل من إجراء جولات استرجاع متعددة — البحث، التحقق، والتنقيح — قبل إنتاج الإجابة النهائية.

في بيئة الإنتاج، ستستبدل قاعدة المعرفة المؤقتة `TRAVEL_KNOWLEDGE_BASE` بفهرس Azure AI Search حقيقي للتعامل مع استرجاع مستندات السفر على نطاق واسع.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى لتحقيق الدقة، يُرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الموثوق به. بالنسبة للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسيرات خاطئة ناتجة عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
